In [ ]:
import numpy as np
from mod3d import BRepBuilderAPI, Render, gp
from pythreejs import (
    BufferGeometry, BufferAttribute, Mesh, Scene, 
    PerspectiveCamera, Renderer, AmbientLight, DirectionalLight,
    MeshBasicMaterial, OrbitControls, EdgesGeometry, LineSegments,
    LineBasicMaterial, MeshPhongMaterial, LineMaterial, Geometry, Line, MeshPhysicalMaterial
)

## Helper Function to Convert OCCT Mesh to pythreejs

In [ ]:
def faces_mesh(faces_data):
    """
    Convert OCCT face tessellation data to a pythreejs mesh.
    
    Parameters:
    -----------
    faces_data : list
        List of tuples (triangles, vertices, normals, uvs) for each face
        
    Returns:
    --------
    mesh : pythreejs.Mesh or None
        The combined mesh for all faces with polygon offset to prevent z-fighting with edges
    """
    if not faces_data:
        return None
    
    # Combine all faces into single mesh
    all_vertices = []
    all_indices = []
    all_normals = []
    vertex_offset = 0
    
    for triangles, vertices, normals, uvs in faces_data:
        # Add vertices
        all_vertices.append(vertices)
        
        # Add normals if available
        if normals is not None:
            all_normals.append(normals)
        
        # Add indices with offset
        all_indices.append(triangles + vertex_offset)
        vertex_offset += vertices.shape[0]
    
    # Concatenate all data
    vertices_combined = np.vstack(all_vertices).astype(np.float32)
    indices_combined = np.vstack(all_indices).astype(np.uint32).ravel()
    
    # Create BufferGeometry
    geometry = BufferGeometry(
        attributes={
            'position': BufferAttribute(vertices_combined, normalized=False),
            'index': BufferAttribute(indices_combined, normalized=False)
        }
    )
    
    # Compute vertex normals for smooth shading
    geometry.exec_three_obj_method('computeVertexNormals')
    
    # Create material with polygon offset to prevent z-fighting with edges
    # polygonOffset pushes the faces back slightly in depth buffer
    material = MeshPhongMaterial(
        color='#2194ce',
        shininess=100,
        side='DoubleSide',
        polygonOffset=True,        # Enable polygon offset
        polygonOffsetFactor=1,     # Offset factor
        polygonOffsetUnits=1       # Offset units
    )

    material = MeshBasicMaterial(
        color='#2194ce',
        side='DoubleSide',
        polygonOffset=True,        # Enable polygon offset
        polygonOffsetFactor=1,     # Offset factor
        polygonOffsetUnits=1       # Offset units
    )

    material = MeshPhysicalMaterial(
        color='#049ef4',              # Bright blue color
        metalness=0.1,                # Low metalness for non-metallic appearance (0-1)
        roughness=1.0,                # Some roughness for realistic surface (0=mirror, 1=diffuse)
        clearcoat=0.5,                # Clear coating layer on top (0-1)
        clearcoatRoughness=0.1,       # Smooth clear coat
        reflectivity=0.5,             # How reflective the surface is (0-1)
        ior=1.5,                      # Index of refraction (glass=1.5, diamond=2.4)
        side='DoubleSide',            # Render both sides of faces
        polygonOffset=True,           # Enable polygon offset to prevent z-fighting with edges
        polygonOffsetFactor=1,        # Offset factor
        polygonOffsetUnits=1          # Offset units
    )
    
    mesh = Mesh(geometry=geometry, material=material)
    mesh.renderOrder = 0  # Render faces first
    return mesh

def edges_mesh(edges_data):
    """
    Convert OCCT edge tessellation data to pythreejs lines.
    
    Parameters:
    -----------
    edges_data : list
        List of tuples (indices, vertices) for each edge
        
    Returns:
    --------
    edge_lines : list or None
        List of pythreejs.Line objects, one per edge, with renderOrder set to appear on top
    """
    if not edges_data:
        return None
    
    # Create separate Line object for each edge
    edge_lines = []
    material = LineBasicMaterial(linewidth=5, color='#000000')
    
    for _, vertices in edges_data:
        # The vertices are already in the correct order for the polyline
        # Just use them directly to create continuous lines
        vertices_array = vertices.astype(np.float32)
        
        # Create geometry with vertices for continuous line
        geometry = Geometry(vertices=vertices_array.tolist())
        
        # Create continuous line for this edge
        line = Line(geometry=geometry, material=material)
        line.renderOrder = 1  # Render edges after faces to ensure visibility
        edge_lines.append(line)
    
    # Return list of edge lines
    return edge_lines

def occt_to_threejs(shape, linear_deflection=0.1, **kwargs):
    """
    Convert an OCCT shape to pythreejs mesh and edge lines.
    
    Parameters:
    -----------
    shape : TopoDS_Shape
        The OCCT shape to convert
    linear_deflection : float, optional
        The maximum distance between the triangulation and the surface (default: 0.1)
        Smaller values create finer meshes
    **kwargs : dict
        Additional arguments passed to extract_tessellation:
        - angle_deflection: Angular deflection in degrees (default: 20.0)
        - relative: Use relative deflection (default: False)
    
    Returns:
    --------
    mesh_face : pythreejs.Mesh
        The rendered mesh for faces
    mesh_edges : list of pythreejs.Line
        List of line objects for edges, rendered on top of faces
        
    Notes:
    ------
    - Faces use polygon offset to prevent z-fighting with edges
    - Edges are rendered as separate continuous polylines per edge
    - RenderOrder ensures edges appear on top of faces
    """
    # Extract tessellation data
    faces_data, edges_data = Render.extract_tessellation(shape, linear_deflection, **kwargs)
    
    mesh_face = faces_mesh(faces_data)
    mesh_edges = edges_mesh(edges_data)

    return mesh_face, mesh_edges

## Example 1: Simple Box

In [ ]:
# Create a box
box = BRepBuilderAPI.MakeBox(10.0, 20.0, 30.0).shape()

# Convert to threejs mesh
mesh_face, mesh_edges = occt_to_threejs(box, linear_deflection=0.5)

camera = PerspectiveCamera(position=[50, 50, 50], aspect=1.5)
# Setup scene (mesh_edges is now a list of lines)
scene_children = [mesh_face, camera, 
                  AmbientLight(intensity=0.5),
                  DirectionalLight(position=[100, 100, 100], intensity=0.5)]
if mesh_edges:
    scene_children.extend(mesh_edges)

scene = Scene(children=scene_children)

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=800, height=600)

from mod3d import TopExp
print(len(TopExp.get_vertices(box)))

renderer

## Example 2: Cylinder with Edges

In [ ]:
# Create a cylinder
cylinder = BRepBuilderAPI.MakeCylinder(5.0, 15.0).shape()

# Convert to mesh
mesh_face, _ = occt_to_threejs(cylinder, linear_deflection=0.2)

# Add edge visualization
edges = EdgesGeometry(mesh_face.geometry)
edge_lines = LineSegments(
    geometry=edges,
    material=LineBasicMaterial(color='black', linewidth=1)
)

# Setup scene
camera = PerspectiveCamera(position=[30, 30, 30], aspect=1.5)
scene = Scene(children=[mesh_face, edge_lines, camera,
                        AmbientLight(intensity=0.5),
                        DirectionalLight(position=[10, 10, 10], intensity=0.5)])

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=800, height=600)
renderer

## Example 3: Sphere

In [ ]:
# Create a sphere
sphere = BRepBuilderAPI.MakeSphere(8.0).shape()

# Convert to mesh with finer deflection
mesh_face, _  = occt_to_threejs(sphere, linear_deflection=0.1, angle_deflection=15.0)

# Setup scene
camera = PerspectiveCamera(position=[25, 25, 25], aspect=8.0/6.0)
scene = Scene(children=[mesh_face, camera,
                        AmbientLight(intensity=0.5),
                        DirectionalLight(position=[10, 10, 10], intensity=0.5)])

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=800, height=600)
renderer

## Example 4: Complex Shape - Filleted Box

In [ ]:
# Create a box with fillets
from mod3d import BRepFillet, TopExp, TopAbs

box = BRepBuilderAPI.MakeBox(20.0, 20.0, 20.0).shape()

# Create fillet
fillet_maker = BRepFillet.MakeFillet(box)

# Add fillets to edges
for edge in TopExp.get_edges(box):
    fillet_maker.add(3.0, edge)  # 3mm radius fillet

filleted_box = fillet_maker.shape()

# Convert to mesh
mesh_face, mesh_edges = occt_to_threejs(filleted_box, linear_deflection=0.2, angle_deflection=10.0)

# Setup scene (mesh_edges is now a list of lines)
scene_children = [ mesh_face, camera,
                  AmbientLight(intensity=0.5),
                  DirectionalLight(position=[10, 10, 10], intensity=0.5)]
if mesh_edges:
    scene_children.extend(mesh_edges)

camera = PerspectiveCamera(position=[50, 50, 50], aspect=1.5)
scene = Scene(children=scene_children)

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=800, height=600)
renderer

## Mesh Quality Comparison

Compare different tessellation quality settings

In [ ]:
sphere = BRepBuilderAPI.MakeSphere(5.0).shape()

# Coarse mesh
mesh_coarse, _ = occt_to_threejs(sphere, linear_deflection=1.0, angle_deflection=30.0)
mesh_coarse.position = [-15, 0, 0]

# Medium mesh
mesh_medium, _ = occt_to_threejs(sphere, linear_deflection=0.3, angle_deflection=15.0)
mesh_medium.position = [0, 0, 0]

# Fine mesh
mesh_fine, _ = occt_to_threejs(sphere, linear_deflection=0.1, angle_deflection=5.0)
mesh_fine.position = [15, 0, 0]

# Setup scene with all three meshes
camera = PerspectiveCamera(position=[0, 20, 40], aspect=2.0)
scene = Scene(children=[mesh_coarse, mesh_medium, mesh_fine, camera,
                        AmbientLight(intensity=0.5),
                        DirectionalLight(position=[10, 10, 10], intensity=0.5)])

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=1200, height=600)
renderer